In [22]:
import ast
import sys
sys.path.append('../../..')

import numpy as np
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv

from src.utils.db import get_connection

load_dotenv()

True

In [23]:
conn = get_connection()
df = pd.read_sql(
    """
    SELECT appid, name_store, genres, positive, negative, release_date
    FROM steam_indie_list
    WHERE genres ILIKE '%Casual%'
    """,
    conn
)
conn.close()

df['total_reviews'] = df['positive'] + df['negative']

# release_date 파싱 후 2024-01-01 이후 필터링
df['release_date_parsed'] = pd.to_datetime(df['release_date'], errors='coerce')
df = df[df['release_date_parsed'] >= '2024-01-01'].reset_index(drop=True)

print(f'Casual 장르 전체 게임 수 (2024-01-01 이후): {len(df):,}개')
print(f'총 리뷰 수: {df["total_reviews"].sum():,}건')
df.head()

/var/folders/0b/g0grvv6j3wgdm_gjyyd_jb7m0000gn/T/ipykernel_49222/3912676115.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


Casual 장르 전체 게임 수 (2024-01-01 이후): 5,982개
총 리뷰 수: 1,505,834건


,appid,name_store,genres,positive,negative,release_date,total_reviews,release_date_parsed
0,2379780,Balatro,"['Casual', 'Indie', 'Strategy']",150524,3042,"20 Feb, 2024",153566,2024-02-20
1,2709570,Supermarket Together,"['Casual', 'Indie', 'Simulation', 'Free To Play']",63994,3441,"9 Aug, 2024",67435,2024-08-09
2,3097560,Liar's Bar,"['Casual', 'Indie', 'Simulation', 'Strategy', ...",44341,4209,"2 Oct, 2024",48550,2024-10-02
3,2567870,Chained Together,"['Adventure', 'Casual', 'Indie', 'Simulation']",48180,4958,"19 Jun, 2024",53138,2024-06-19
4,2670630,Supermarket Simulator,"['Casual', 'Indie', 'Simulation']",64104,4237,"19 Jun, 2025",68341,2025-06-19


### 데이터 로드

`steam_indie_list`에서 `genres ILIKE '%Casual%'` 조건으로 필터링했다.
Casual은 Steam에서 메타 장르로 쓰이는 경우가 많아 게임 수가 많고,
리뷰 수 분포는 롱테일 구조일 것으로 예상된다.

In [24]:
MIN_REVIEWS = 30  # 신뢰도 확보를 위한 최소 리뷰 수
Z = 1.96          # 95% 신뢰구간

def wilson_lower_bound(positive: int, total: int, z: float = 1.96) -> float:
    """Wilson Score 하한값 계산. total < MIN_REVIEWS면 NaN 반환."""
    if total < MIN_REVIEWS:
        return float('nan')
    p = positive / total
    z2 = z ** 2
    numerator = p + z2 / (2 * total) - z * np.sqrt(p * (1 - p) / total + z2 / (4 * total ** 2))
    denominator = 1 + z2 / total
    return numerator / denominator

df['wilson_score'] = df.apply(
    lambda r: wilson_lower_bound(r['positive'], r['total_reviews']), axis=1
)

df_valid = df.dropna(subset=['wilson_score']).copy()

print(f'Wilson Score 계산 가능 게임 (리뷰 >= {MIN_REVIEWS}): {len(df_valid):,}개')
print(f'제외된 게임 (리뷰 < {MIN_REVIEWS}): {len(df) - len(df_valid):,}개')
print(f'\nWilson Score 분포:')
print(df_valid['wilson_score'].describe().round(4))

Wilson Score 계산 가능 게임 (리뷰 >= 30): 1,714개
제외된 게임 (리뷰 < 30): 4,268개

Wilson Score 분포:
count    1714.0000
mean        0.7740
std         0.1537
min         0.0793
25%         0.6986
50%         0.8130
75%         0.8887
max         0.9946
Name: wilson_score, dtype: float64


### Wilson Score

Wilson Score 하한값은 "이 리뷰 수로 볼 때 실제 긍정률이 최소 얼마 이상임을 95% 신뢰할 수 있는가"를 나타낸다.

- 리뷰가 적은 게임은 긍정률이 높아도 하한값이 낮게 나온다 — 신뢰도가 낮기 때문이다.
- 리뷰가 많고 긍정률이 높은 게임이 자연스럽게 상위권에 위치한다.
- `total_reviews < 30`인 게임은 통계적으로 의미 있는 평가가 불가능하므로 제외했다.

In [25]:
q1 = df_valid['total_reviews'].quantile(0.25)
q3 = df_valid['total_reviews'].quantile(0.75)

print(f'=== Casual 장르 리뷰 수 사분위수 (리뷰 >= {MIN_REVIEWS} 기준) ===')
print(df_valid['total_reviews'].describe(percentiles=[.25, .5, .75]).round(0).to_string())
print(f'\n구간 경계:')
print(f'  소형: {MIN_REVIEWS} ~ {int(q1)}건')
print(f'  중형: {int(q1)+1} ~ {int(q3)}건')
print(f'  대형: {int(q3)+1}건 이상')

=== Casual 장르 리뷰 수 사분위수 (리뷰 >= 30 기준) ===
count      1714.0
mean        859.0
std        5379.0
min          30.0
25%          48.0
50%         100.0
75%         294.0
max      153566.0

구간 경계:
  소형: 30 ~ 48건
  중형: 49 ~ 294건
  대형: 295건 이상


### 리뷰 수 구간 설정

사분위수(Q1, Q3)를 경계로 소형/중형/대형 3구간을 나눈다.
자의적인 기준이 아닌 Casual 장르 실제 분포에 근거한 구간이므로, 각 구간에 게임이 균등하게 분포한다.

- **소형**: 리뷰 수가 적어 인지도가 낮거나 출시 초기인 게임
- **중형**: 시장에서 어느 정도 반응을 얻은 일반적인 인디게임
- **대형**: 팬덤이 형성되거나 장기 흥행에 성공한 게임

In [26]:
N = 5  # 구간별 상위/하위 각 추출 수

def assign_size_tier(total: int, q1: float, q3: float) -> str:
    if total <= q1:
        return '소형'
    elif total <= q3:
        return '중형'
    return '대형'

df_valid['size_tier'] = df_valid['total_reviews'].apply(
    lambda x: assign_size_tier(x, q1, q3)
)

tiers = ['소형', '중형', '대형']
records = []

for tier in tiers:
    group = df_valid[df_valid['size_tier'] == tier].sort_values('wilson_score', ascending=False)
    top = group.head(N).copy()
    top['selection'] = '상위'
    bottom = group.tail(N).copy()
    bottom['selection'] = '하위'
    records.append(top)
    records.append(bottom)

df_selected = pd.concat(records).reset_index(drop=True)

cols = ['size_tier', 'selection', 'name_store', 'total_reviews', 'positive', 'negative', 'wilson_score']
for tier in tiers:
    for sel in ['상위', '하위']:
        subset = df_selected[(df_selected['size_tier'] == tier) & (df_selected['selection'] == sel)]
        print(f'=== {tier} {sel} {N}개 ===')
        print(subset[cols].to_string(index=False))
        print()

=== 소형 상위 5개 ===
size_tier selection                               name_store  total_reviews  positive  negative  wilson_score
       소형        상위 Nekokami - The Human Restoration Project             48        48         0      0.925897
       소형        상위                                 ハチバンバスピス             46        46         0      0.922924
       소형        상위                               GooGooRise             45        45         0      0.921346
       소형        상위                            Space Sprouts             45        45         0      0.921346
       소형        상위                      I KNOW YOUR ADDRESS             44        44         0      0.919702

=== 소형 하위 5개 ===
size_tier selection                    name_store  total_reviews  positive  negative  wilson_score
       소형        하위                       Obscula             37        11        26      0.174893
       소형        하위 Wizlite: Everybody loved RPGs             33        10        23      0.173753
       소

### 상위/하위 게임 선별 해석

- **상위 게임**: 리뷰 수가 충분하면서 긍정률이 높은 게임이다. Casual 장르에서 유저 만족도와 대중성을 동시에 확보한 성공 사례로 볼 수 있다.
- **하위 게임**: 리뷰 수는 있지만 부정 반응이 강하거나, 리뷰가 적어 Wilson Score 하한이 낮게 형성된 게임이다. 부정 리뷰 원인 분석의 주요 샘플이 된다.

**인디 개발사를 위한 제안:** 상위 게임의 장르 태그, 가격대, 출시 시기를 분석하면 Casual 장르에서 유저 만족도를 높이는 공통 패턴을 도출할 수 있다.

In [27]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

tier_order = ['소형', '중형', '대형']
colors = {'상위': '#2196F3', '하위': '#F44336'}

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=tier_order,
    vertical_spacing=0.12,
)

for i, tier in enumerate(tier_order, 1):
    subset = df_selected[df_selected['size_tier'] == tier].sort_values('wilson_score')
    for sel in ['하위', '상위']:
        group = subset[subset['selection'] == sel]
        fig.add_trace(
            go.Bar(
                x=group['wilson_score'],
                y=group['name_store'],
                orientation='h',
                name=sel,
                marker_color=colors[sel],
                showlegend=(i == 1),
            ),
            row=i, col=1,
        )

fig.update_xaxes(range=[0, 1])
fig.update_layout(
    title='Casual 장르 구간별 Wilson Score 상위/하위 5개 게임',
    height=900,
    barmode='group',
    legend_title='구분',
)
fig.show()

# 구간별 Wilson Score 분포 박스플롯
fig2 = px.box(
    df_valid,
    x='size_tier',
    y='wilson_score',
    color='size_tier',
    category_orders={'size_tier': tier_order},
    title='Casual 장르 — 구간별 Wilson Score 분포',
    labels={'size_tier': '규모 구간', 'wilson_score': 'Wilson Score'},
    points='outliers',
    height=450,
)
fig2.update_layout(showlegend=False)
fig2.show()

### 시각화 해석

- **가로 막대 차트**: 상위/하위 게임의 Wilson Score 격차가 크다면, 장르 내 게임 품질 편차가 심하다는 신호다.
- **산점도**: 리뷰 수가 적은 구간(좌측)에서 Wilson Score가 낮게 형성되는 경향이 뚜렷하게 나타난다. 이는 Wilson Score가 신뢰도를 반영하기 때문이며, 리뷰 수가 늘수록 실제 긍정률에 수렴하는 구조다.

In [28]:
output_path = '../../../data/processed/casual_wilson_selected.csv'
df_selected[['appid', 'name_store', 'size_tier', 'selection', 'positive', 'negative', 'total_reviews', 'wilson_score']].to_csv(
    output_path, index=False
)
print(f'저장 완료: {output_path}')
print(f'총 {len(df_selected)}개 게임 (구간 3개 × 상위/하위 {N}개씩 × 2 = {3*N*2}개)')

저장 완료: ../../../data/processed/casual_wilson_selected.csv
총 30개 게임 (구간 3개 × 상위/하위 5개씩 × 2 = 30개)
